======================================================================================================================

$$\text{This script plots a reduced Sobol indices with independently scrambled Sobol designs}$$
$$\text{Avulundiah Edwin Phiri}$$
$$\text{Department of Mathematics, Copperbelt University, Kitwe, Zambia}$$
$$\text{04/08/2026}$$

======================================================================================================================

In [4]:
"""Reproduce reduced Sobol indices with independently scrambled Sobol designs.
Dependencies: numpy, scipy, matplotlib. Twelve independent scrambled designs
of N=32768 rows are used to calculate error bars across randomizations.
Inputs: r=delta/gamma with delta~U(.1,.5), gamma~U(.5,1.5) independent;
kappa~U(.15,.45), lambda~U(.2,.45), n~U(4,12) independent.
The full Sobol input dimension is four; the underlying delta/gamma pair
is generated separately in each pick-freeze half to preserve its induced law.
"""
from pathlib import Path
from csv import DictWriter
import numpy as np
from scipy.stats import qmc, t as student_t
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

OUT=Path(__file__).resolve().parent
M=15; N=2**M; R=12; X0=0.0
names=('r','kappa','lambda','n')

def transform(u):
    P=np.empty((len(u),4))
    P[:,0]=(.1+.4*u[:,0])/(.5+u[:,1])
    P[:,1]=.15+.30*u[:,2]
    P[:,2]=.20+.25*u[:,3]
    P[:,3]=4+8*u[:,4]
    return P

def eqK(P):
    r,kappa,lam,n=P.T
    s=lam*kappa/(1-lam)
    H=r*(1-s)/lam
    c=1+X0
    lo=np.zeros_like(H);hi=np.full_like(H,c)
    for _ in range(65):
        mid=(lo+hi)*.5
        positive=mid-H*np.maximum(c-mid,0)**n>0
        hi=np.where(positive,mid,hi)
        lo=np.where(positive,lo,mid)
    return (c-(lo+hi)*.5)**n

est=[]
for i in range(R):
    # u dimension 10: 5 to construct one 4-var sample A, and 5 for B
    uv=qmc.Sobol(d=10,scramble=True,seed=20261121+i).random_base2(M)
    A=transform(uv[:,:5]); B=transform(uv[:,5:])
    YA=eqK(A); YB=eqK(B); V=np.var(np.r_[YA,YB]);
    S1=np.empty(4);ST=np.empty(4)
    for j in range(4):
        C=A.copy();C[:,j]=B[:,j];YC=eqK(C)
        S1[j]=1-np.mean((YB-YC)**2)/(2*V)
        ST[j]=np.mean((YA-YC)**2)/(2*V)
    est.append(np.c_[S1,ST]); print(i, np.round(S1,6),np.round(ST,6), flush=True)
est=np.stack(est)
mean=est.mean(0)
se=est.std(0,ddof=1)/np.sqrt(R)
ci=student_t.ppf(.975,R-1)*se
rows=[]
for j,nam in enumerate(names):
    rows.append(dict(Parameter=nam,FirstOrder=mean[j,0],FirstCI_L=mean[j,0]-ci[j,0],FirstCI_U=mean[j,0]+ci[j,0],TotalOrder=mean[j,1],TotalCI_L=mean[j,1]-ci[j,1],TotalCI_U=mean[j,1]+ci[j,1]))
with (OUT/'global_sensitivity_indices_reduced.csv').open('w',newline='') as f:
    w=DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
fig,ax=plt.subplots(figsize=(7.5,3.8))
x=np.arange(4);width=.35
ax.bar(x-width/2,mean[:,0],width,yerr=ci[:,0],capsize=3,label='First order')
ax.bar(x+width/2,mean[:,1],width,yerr=ci[:,1],capsize=3,label='Total order')
ax.set_xticks(x,[r'$r=\delta/\gamma$',r'$\kappa$',r'$\lambda$',r'$n$'])
ax.set_ylim(0,.77);ax.set_ylabel('Sobol index')
ax.legend(frameon=False);ax.grid(axis='y',alpha=.2)
fig.tight_layout();fig.savefig(OUT/'global_sensitivity_reduced.png',dpi=350);fig.savefig(OUT/'global_sensitivity_reduced.pdf');plt.close(fig)
print('summary',rows)




NameError: name '__file__' is not defined